# 🎬 MimicMotion — Versión Gradio (HuggingFace Spaces)

Este notebook replica la lógica del `app.py` diseñado para **HuggingFace Spaces** con Gradio.
Anima una **foto** usando el movimiento de un **video de referencia**, basado en [MimicMotion (Tencent)](https://github.com/tencent/MimicMotion).

## Diferencias respecto al notebook Colab estándar
- Usa exactamente las mismas versiones de librerías del `requirements.txt` de Spaces
- Incluye el pre-trim de video (`trim_video_to_budget`) antes del bloque GPU
- Replica la función `setup()` del `app.py` (clonado + modelos + parche loader)
- Interfaz Gradio ejecutable localmente

---
> **GPU requerida**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU**

## Paso 0 — Verificar GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else
      '❌ GPU no detectada. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU')

## Paso 1 — Instalar dependencias

Versiones compatibles con Python 3.12 y Colab 2025:
```
diffusers==0.32.2 | transformers>=4.46  | decord==0.6.0
einops==0.8.0     | omegaconf==2.3.0    | accelerate==0.32.0
onnxruntime==1.18.0 | av==12.2.0        | opencv-python>=4.10.0
```
> **Nota:** `transformers==4.42.0` del requirements.txt original no exporta `EncoderDecoderCache`,
> requerido por `diffusers==0.32.2`. Se usa `transformers>=4.46` para compatibilidad.

In [ ]:
import sys, subprocess

result = subprocess.run(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
    capture_output=True, text=True
)
numpy_ver_str = result.stdout.strip()
print(f'NumPy actual: {numpy_ver_str}')

if numpy_ver_str and int(numpy_ver_str.split('.')[0]) >= 2:
    print('Bajando NumPy a 1.26.4 ...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4', '--force-reinstall'],
        check=True
    )
    print('NumPy 1.26.4 instalado — reiniciando kernel...')
    import os; os.kill(os.getpid(), 9)
else:
    print(f'NumPy {numpy_ver_str} OK, continua.')

In [ ]:
import sys, subprocess

print('Instalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.32.2',
    # transformers>=4.46 requerido: 4.42 no exporta EncoderDecoderCache que diffusers==0.32.2 necesita
    'transformers>=4.46,<5.0',
    'tokenizers>=0.19',
    'decord==0.6.0',
    'einops==0.8.0',
    'omegaconf==2.3.0',
    'opencv-python-headless>=4.10.0',
    'matplotlib==3.9.0',
    'onnxruntime==1.18.0',
    'accelerate==0.32.0',
    'av==12.2.0',
    'pyyaml',
    'huggingface_hub',
    'torch', 'torchvision',
    'Pillow',
    'gradio>=5.0',
], check=True)
print('Dependencias instaladas')

import numpy as np, torch, transformers, diffusers
print(f'NumPy       : {np.__version__}')
print(f'PyTorch     : {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'transformers: {transformers.__version__}')
print(f'diffusers   : {diffusers.__version__}')

# Verificar que EncoderDecoderCache existe (requerido por diffusers 0.32.2)
from transformers import EncoderDecoderCache
print('EncoderDecoderCache: OK')

## Paso 2 — Setup (replica `setup()` del app.py)

Esta función hace exactamente lo mismo que el `setup()` del Space:
1. Clona el repositorio MimicMotion
2. Parchea `loader.py` para PyTorch moderno
3. Descarga modelos DWPose
4. Descarga checkpoint MimicMotion_1-1.pth
5. Descarga SVD (requiere HF_TOKEN con acceso aceptado)

In [ ]:
import os, sys, subprocess, logging
import huggingface_hub

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

HF_TOKEN   = os.environ.get('HF_TOKEN', '')
MIMIC_DIR  = '/content/MimicMotion'
MODELS_DIR = f'{MIMIC_DIR}/models'
SVD_DIR    = f'{MODELS_DIR}/SVD'
DWPOSE_DIR = f'{MODELS_DIR}/DWPose'
MAX_OUTPUT_FRAMES = 48


def setup():
    if not os.path.exists(MIMIC_DIR):
        logger.info('Clonando tencent/MimicMotion ...')
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/tencent/MimicMotion.git', MIMIC_DIR],
            check=True,
        )
    sys.path.insert(0, MIMIC_DIR)

    loader_path = os.path.join(MIMIC_DIR, 'mimicmotion/utils/loader.py')
    if os.path.exists(loader_path):
        with open(loader_path) as f:
            content = f.read()
        if 'safe_globals(*allowed_modules)' in content:
            logger.info('Parcheando loader.py para PyTorch moderno ...')
            content = content.replace(
                'safe_globals(*allowed_modules)',
                'safe_globals(allowed_modules)',
            )
            with open(loader_path, 'w') as f:
                f.write(content)

    os.makedirs(MODELS_DIR, exist_ok=True)
    os.makedirs(DWPOSE_DIR, exist_ok=True)

    for fname in ['yolox_l.onnx', 'dw-ll_ucoco_384.onnx']:
        dst = os.path.join(DWPOSE_DIR, fname)
        if not os.path.exists(dst):
            logger.info(f'Descargando DWPose: {fname}')
            huggingface_hub.hf_hub_download(
                repo_id='yzd-v/DWPose', filename=fname, local_dir=DWPOSE_DIR
            )

    mimic_weight = os.path.join(MODELS_DIR, 'MimicMotion_1-1.pth')
    if not os.path.exists(mimic_weight):
        logger.info('Descargando MimicMotion_1-1.pth ...')
        huggingface_hub.hf_hub_download(
            repo_id='tencent/MimicMotion',
            filename='MimicMotion_1-1.pth',
            local_dir=MODELS_DIR,
        )

    if not os.path.exists(os.path.join(SVD_DIR, 'model_index.json')):
        if HF_TOKEN:
            logger.info('Descargando stable-video-diffusion-img2vid-xt-1-1 ...')
            huggingface_hub.snapshot_download(
                repo_id='stabilityai/stable-video-diffusion-img2vid-xt-1-1',
                local_dir=SVD_DIR,
                token=HF_TOKEN,
                ignore_patterns=['*.bin'],
            )
        else:
            logger.warning('HF_TOKEN no configurado — modelo SVD no disponible.')

    logger.info('Setup completo.')


setup()

## Paso 3 — Autenticación HuggingFace

Necesitas aceptar los términos en [stabilityai/stable-video-diffusion-img2vid-xt-1-1](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt-1-1) y proporcionar tu token.

In [ ]:
import os
from huggingface_hub import login, whoami

HF_TOKEN = ''  # @param {type:"string"}

if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    try:
        info = whoami()
        print(f'Autenticado como: {info["name"]}')
    except Exception as e:
        print(f'Token aceptado pero no se pudo verificar: {e}')
    setup()
else:
    print('Sin token — el modelo SVD no podra descargarse.')

## Paso 4 — Subir archivos de entrada

In [ ]:
from google.colab import files
import os

os.makedirs('/content/inputs', exist_ok=True)

print('Sube la FOTO del personaje (jpg/png):')
uploaded_img = files.upload()

ref_image_path = None
for fname in uploaded_img:
    dest = f'/content/inputs/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded_img[fname])
    ref_image_path = dest
    print(f'Foto: {ref_image_path}')

In [ ]:
print('Sube el VIDEO DE REFERENCIA (mp4):')
uploaded_vid = files.upload()

ref_video_path = None
for fname in uploaded_vid:
    dest = f'/content/inputs/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded_vid[fname])
    ref_video_path = dest
    print(f'Video: {ref_video_path}')

In [ ]:
from PIL import Image
import IPython.display as ipd
import cv2

assert ref_image_path and os.path.exists(ref_image_path), 'Foto no encontrada.'
assert ref_video_path and os.path.exists(ref_video_path), 'Video no encontrado.'

img = Image.open(ref_image_path)
print(f'Foto : {img.size[0]}x{img.size[1]} px')
ipd.display(img.resize((256, int(256 * img.size[1] / img.size[0]))))

cap = cv2.VideoCapture(ref_video_path)
total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_v   = cap.get(cv2.CAP_PROP_FPS)
w_v     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_v     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {w_v}x{h_v} | {total_f} frames | {fps_v:.1f} fps | {total_f/fps_v:.1f}s')

## Paso 5 — Configurar parámetros

Los mismos parámetros expuestos en la UI Gradio del `app.py`.

In [ ]:
RESOLUTION          = 576    # @param {type:"slider", min:256, max:768, step:64}
NUM_FRAMES          = 16     # @param {type:"slider", min:8, max:72, step:8}
NUM_INFERENCE_STEPS = 20     # @param {type:"slider", min:5, max:50, step:1}
NOISE_AUG_STRENGTH  = 0.0563 # @param {type:"number"}
GUIDANCE_SCALE      = 2.0    # @param {type:"number"}
SAMPLE_STRIDE       = 4      # @param {type:"slider", min:1, max:4, step:1}
SEED                = 42     # @param {type:"integer"}

print('Configuracion:')
print(f'  Resolucion          : {RESOLUTION}px')
print(f'  Frames por tile     : {NUM_FRAMES}')
print(f'  Pasos de inferencia : {NUM_INFERENCE_STEPS}')
print(f'  Noise aug strength  : {NOISE_AUG_STRENGTH}')
print(f'  Guidance scale      : {GUIDANCE_SCALE}')
print(f'  Sample stride       : {SAMPLE_STRIDE}')
print(f'  Semilla             : {SEED}')
print(f'  Max frames salida   : {MAX_OUTPUT_FRAMES}')

## Paso 6 — Pre-trim del video (lógica del app.py)

Recorta el video con `ffmpeg` ANTES del bloque GPU.
Réplica exacta de `trim_video_to_budget()` del `app.py`.

In [ ]:
import tempfile, subprocess


def trim_video_to_budget(video_path, max_output_frames, sample_stride):
    try:
        import decord
        vr = decord.VideoReader(video_path)
        total = len(vr)
        max_raw = (max_output_frames + 1) * sample_stride
        if total <= max_raw:
            logger.info(f'Video {total} frames <= budget {max_raw}, sin recorte.')
            return video_path
        fps = vr.get_avg_fps() or 30.0
        duration = max_raw / fps
        out_path = tempfile.mktemp(suffix='.mp4')
        subprocess.run(
            ['ffmpeg', '-y', '-i', video_path,
             '-t', f'{duration:.3f}',
             '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
             '-an', out_path],
            check=True, capture_output=True,
        )
        logger.info(f'Pre-trim: {total} -> {max_raw} frames ({duration:.1f}s)')
        return out_path
    except Exception as e:
        logger.warning(f'Video pre-trim fallo ({e}), usando original.')
        return video_path


ref_video_trimmed = trim_video_to_budget(ref_video_path, MAX_OUTPUT_FRAMES, int(SAMPLE_STRIDE))
print(f'Video para inferencia: {ref_video_trimmed}')

## Paso 7 — Inferencia GPU (lógica de `run_mimicmotion()` del app.py)

Replica el bloque `@spaces.GPU` del Space.

In [ ]:
import torch
from omegaconf import OmegaConf
import yaml

os.chdir(MIMIC_DIR)
if MIMIC_DIR not in sys.path:
    sys.path.insert(0, MIMIC_DIR)

import huggingface_hub as _hfh
if not hasattr(_hfh, 'cached_download'):
    _hfh.cached_download = _hfh.hf_hub_download
    print('Patch cached_download aplicado')

import accelerate.utils.memory as _aum
if not hasattr(_aum, 'clear_device_cache'):
    def _cdc():
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    _aum.clear_device_cache = _cdc
    import accelerate.utils as _au
    _au.clear_device_cache = _cdc
    print('Patch clear_device_cache aplicado')


def run_mimicmotion(
    ref_image_path, ref_video_path,
    resolution=RESOLUTION, num_frames=NUM_FRAMES,
    num_inference_steps=NUM_INFERENCE_STEPS,
    noise_aug_strength=NOISE_AUG_STRENGTH,
    guidance_scale=GUIDANCE_SCALE,
    sample_stride=SAMPLE_STRIDE, seed=SEED,
):
    from mimicmotion.utils.utils import save_to_mp4
    from inference import preprocess, run_pipeline
    from mimicmotion.utils.geglu_patch import patch_geglu_inplace
    patch_geglu_inplace()
    from mimicmotion.utils.loader import create_pipeline

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logger.info(f'Cargando pipeline en {device} ...')

    cfg_dict = {
        'base_model_path': SVD_DIR,
        'ckpt_path': os.path.join(MODELS_DIR, 'MimicMotion_1-1.pth'),
    }
    with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False) as f:
        yaml.dump(cfg_dict, f)
        cfg_path = f.name

    infer_config = OmegaConf.load(cfg_path)
    torch.set_default_dtype(torch.float16)
    pipe = create_pipeline(infer_config, device)
    logger.info('Pipeline cargado. Ejecutando DWPose ...')

    pose_pixels, image_pixels = preprocess(
        ref_video_path, ref_image_path,
        resolution=resolution,
        sample_stride=sample_stride,
    )

    if pose_pixels.shape[0] > MAX_OUTPUT_FRAMES + 1:
        logger.info(f'Recortando pose: {pose_pixels.shape[0]} -> {MAX_OUTPUT_FRAMES + 1}')
        pose_pixels = pose_pixels[: MAX_OUTPUT_FRAMES + 1]

    actual_frames = pose_pixels.shape[0]
    if num_frames > actual_frames:
        logger.info(f'Ajustando num_frames {num_frames} -> {actual_frames}')
        num_frames = actual_frames

    task_config = OmegaConf.create({
        'num_frames':          num_frames,
        'frames_overlap':      4,
        'num_inference_steps': num_inference_steps,
        'noise_aug_strength':  noise_aug_strength,
        'guidance_scale':      guidance_scale,
        'seed':                seed,
        'resolution':          resolution,
        'sample_stride':       sample_stride,
    })

    video_frames = run_pipeline(pipe, image_pixels, pose_pixels, device, task_config)
    out_path = tempfile.mktemp(suffix='.mp4')
    save_to_mp4(video_frames, out_path, fps=15)
    logger.info(f'Video generado: {out_path}')
    return out_path


print('Funcion run_mimicmotion() definida.')

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:256'

print('Iniciando inferencia...')
print(f'   Foto  : {ref_image_path}')
print(f'   Video : {ref_video_trimmed}')
print('-' * 60)

output_video_path = run_mimicmotion(
    ref_image_path      = ref_image_path,
    ref_video_path      = ref_video_trimmed,
    resolution          = int(RESOLUTION),
    num_frames          = int(NUM_FRAMES),
    num_inference_steps = int(NUM_INFERENCE_STEPS),
    noise_aug_strength  = float(NOISE_AUG_STRENGTH),
    guidance_scale      = float(GUIDANCE_SCALE),
    sample_stride       = int(SAMPLE_STRIDE),
    seed                = int(SEED),
)

print(f'Video de salida: {output_video_path}')

## Paso 8 — Ver y descargar el resultado

In [ ]:
from IPython.display import HTML, display as ipy_display
from base64 import b64encode

assert os.path.exists(output_video_path), 'No se encontro el video de salida.'
size_mb = os.path.getsize(output_video_path) / 1e6
print(f'Video: {output_video_path} ({size_mb:.1f} MB)')

with open(output_video_path, 'rb') as f:
    b64 = b64encode(f.read()).decode()

ipy_display(HTML(
    f'<video controls width="480" autoplay loop>'
    f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
    f'</video>'
))

In [ ]:
from google.colab import files
print('Descargando video...')
files.download(output_video_path)

## (Opcional) Lanzar la interfaz Gradio

Replica el `demo.launch()` del `app.py`. En Colab usa `share=True` para un enlace publico.

In [ ]:
import gradio as gr


def generate_gradio(
    ref_image, ref_video,
    resolution, num_frames, num_inference_steps,
    noise_aug_strength, guidance_scale,
    sample_stride, seed
):
    if ref_image is None:
        raise gr.Error('Por favor sube una imagen de referencia.')
    if ref_video is None:
        raise gr.Error('Por favor sube un video de conduccion.')
    if isinstance(ref_video, dict):
        ref_video = ref_video.get('video') or ref_video.get('name') or ref_video.get('path')
    if isinstance(ref_image, dict):
        ref_image = ref_image.get('path') or ref_image.get('name')
    ref_video = trim_video_to_budget(ref_video, MAX_OUTPUT_FRAMES, int(sample_stride))
    return run_mimicmotion(
        ref_image, ref_video,
        int(resolution), int(num_frames), int(num_inference_steps),
        float(noise_aug_strength), float(guidance_scale),
        int(sample_stride), int(seed),
    )


with gr.Blocks(title='MimicMotion') as demo:
    gr.Markdown(
        '# MimicMotion\n'
        'Sube una **imagen de referencia** y un **video de conduccion**.\n\n'
        '> Tips: usa videos cortos (3-5s), stride=4, ~3-4 min de generacion.'
    )
    with gr.Row():
        with gr.Column():
            ref_image_input  = gr.Image(label='Imagen de referencia', type='filepath')
            ref_video_input  = gr.Video(label='Video de conduccion')
            with gr.Accordion('Configuracion avanzada', open=False):
                resolution_sl    = gr.Slider(256, 768,  value=576,    step=64,    label='Resolucion')
                num_frames_sl    = gr.Slider(8,   72,   value=16,     step=8,     label='Frames por tile')
                num_steps_sl     = gr.Slider(5,   50,   value=20,     step=1,     label='Pasos de inferencia')
                noise_aug_sl     = gr.Slider(0.0, 0.1,  value=0.0563, step=0.001, label='Noise aug strength')
                guidance_sl      = gr.Slider(1.0, 10.0, value=2.0,    step=0.5,   label='Guidance scale')
                sample_stride_sl = gr.Slider(1,   4,    value=4,      step=1,     label='Sample stride')
                seed_input       = gr.Number(value=42, label='Semilla', precision=0)
            run_btn = gr.Button('Generar', variant='primary')
        with gr.Column():
            output_video = gr.Video(label='Video de salida', autoplay=True)
    run_btn.click(
        fn=generate_gradio,
        inputs=[ref_image_input, ref_video_input, resolution_sl, num_frames_sl, num_steps_sl,
                noise_aug_sl, guidance_sl, sample_stride_sl, seed_input],
        outputs=output_video,
    )

demo.launch(share=True)

---
## Referencia de parámetros

| Parámetro | Descripción | Valor por defecto |
|---|---|---|
| `resolution` | Alto de salida en píxeles | 576 |
| `num_frames` | Frames por tile del pipeline | 16 |
| `num_inference_steps` | Pasos del modelo de difusión | 20 |
| `noise_aug_strength` | Variación adicional | 0.0563 |
| `guidance_scale` | Fuerza de seguimiento de la pose | 2.0 |
| `sample_stride` | Intervalo de muestreo del video | 4 |
| `seed` | Semilla para reproducibilidad | 42 |
| `MAX_OUTPUT_FRAMES` | Límite global de frames | 48 |

### Flujo de datos (app.py)
```
Entrada → trim_video_to_budget() → run_mimicmotion()
                                        ├── create_pipeline()   (SVD + MimicMotion)
                                        ├── preprocess()        (DWPose)
                                        ├── pose_pixels[:49]    (recorte)
                                        ├── run_pipeline()      (difusion)
                                        └── save_to_mp4()       (salida)
```